# ETL Cleaning — Order Reviews Table
**Source:** `olist_order_reviews_dataset.csv`  
**Output:** `data/cleaned/order_reviews_cleaned.csv`

### Key characteristics of this table
- Each row represents one customer review for one order
- `review_comment_title` and `review_comment_message` are optional — high missing rate is expected
- `review_score` is an integer from 1 to 5

### Cleaning Steps
1. Load raw data
2. Initial inspection (shape, columns, missing values, distributions)
3. Investigate duplicate review_id
4. Remove duplicate rows
5. Convert date columns to datetime
6. Export cleaned data

## Step 1 — Load Raw Data

In [8]:
import pandas as pd

# ── 1. Load raw data ──────────────────────────────────────────
df = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

## Step 2 — Initial Inspection

Note: high missing rates in `review_comment_title` and `review_comment_message` are expected —  
most customers only submit a score without writing a comment.

In [9]:
# ── 2. Initial inspection ─────────────────────────────────────
print("=== Shape ===")
print(df.shape)

print("\n=== Column names ===")
print(df.columns.tolist())

print("\n=== First 5 rows ===")
print(df.head())

print("\n=== Current dtypes ===")
print(df.dtypes)

print("\n=== Missing values ===")
print(df.isnull().sum())

print("\n=== Numeric columns summary ===")
print(df.describe())

print("\n=== Categorical columns summary ===")
print(df.describe(include='str'))

=== Shape ===
(99224, 7)

=== Column names ===
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

=== First 5 rows ===
                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                              

## Step 3 — Investigate Duplicate review_id

From initial inspection, `unique review_id` (98,410) is less than total rows (99,224),  
meaning some `review_id` values appear more than once.

We investigate whether the same `review_id` maps to different `order_id` values,  
which would indicate a data system error rather than legitimate duplicate reviews.

In [10]:
# ── 3a. Count duplicates ──────────────────────────────────────
print(f"Duplicate review_id: {df.duplicated(subset=['review_id']).sum()}")
print(f"Duplicate order_id: {df.duplicated(subset=['order_id']).sum()}")

# keep=False marks ALL occurrences of a duplicate as True (including the first)
# so we can see both copies side by side
dup_review = df[df.duplicated(subset=['review_id'], keep=False)]
print(f"\n=== Sample duplicated review_id rows ===")
print(dup_review.head(6))

Duplicate review_id: 814
Duplicate order_id: 551

=== Sample duplicated review_id rows ===
                            review_id                          order_id  \
200  28642ce6250b94cc72bc85960aec6c62  e239d280236cdd3c40cb2c033f681d1c   
344  a0a641414ff718ca079b3967ef5c2495  169d7e0fd71d624d306f132acd791cbe   
346  f4d74b17cd63ee35efa82cd2567de911  f269e83a82f64baa3de97c2ebf3358f6   
360  ecbaf1fce7d2c09bfab46f89065afeaf  2451b9756f310d4cff5c7987b393870d   
393  6b1de94de0f4bd84dfc4136818242faa  92acf87839903a94aeca0e5040d99acb   
433  957011305e7a4b6c8a266eeeb8e0316d  54b4da510fed5dc3cf3e7a8e50a5f224   

     review_score review_comment_title  \
200             5                  NaN   
344             5                  NaN   
346             3                  NaN   
360             5                  NaN   
393             5                  NaN   
433             4                  NaN   

                                review_comment_message review_creation_date  \
200      

In [11]:
# ── 3b. Check if same review_id maps to different order_id ────
# If yes, this is a data system error — the same review was assigned to multiple orders
dup = df[df.duplicated(subset=['review_id'], keep=False)]
print(dup[['review_id', 'order_id', 'review_score', 'review_creation_date']].sort_values('review_id').head(10))

                              review_id                          order_id  \
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   
57280  0174caf0ee5964646040cd94e15ac95e  74db91e33b4e1fd865356c89a61abf1f   
54832  017808d29fd1f942d97e50184dfb4c13  8daaa9e99d60fbba579cc1c3e3bfae01   
99167  017808d29fd1f942d97e50184dfb4c13  b1461c8882153b5fe68307c46a506e39   
20621  0254bd905dc677a6078990aad3331a36  5bf226cf882c5bf4247f89a97c86f273   
96080  0254bd905dc677a6078990aad3331a36  331b367bdd766f3d1cf518777317b5d9   

       review_score review_creation_date  
46678             1  2018-03-07 00:00:00  
29841             1  2018-03-07 00:00:00  
90677             5  20

## Step 4 — Remove Duplicate Rows

Confirmed: same `review_id` maps to different `order_id` values with identical scores and dates.  
This is a data system error — the same review was incorrectly assigned to multiple orders.  
Since there is no basis for choosing which `order_id` is correct, we keep the first occurrence and drop the rest.

In [12]:
# ── 4. Remove duplicate review_id, keep first occurrence ──────
rows_before = len(df)
df = df.drop_duplicates(subset=['review_id'], keep='first')
rows_after = len(df)

print(f"Rows removed: {rows_before - rows_after}")
print(f"Rows remaining: {rows_after}")

Rows removed: 814
Rows remaining: 98410


## Step 5 — Convert Date Columns to Datetime

In [13]:
# ── 5. Convert date columns to datetime ──────────────────────
date_cols = [
    'review_creation_date',
    'review_answer_timestamp'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print("=== Dtypes after conversion ===")
print(df[date_cols].dtypes)

=== Dtypes after conversion ===
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


## Step 6 — Export Cleaned Data

In [14]:
# ── 6. Export cleaned data ────────────────────────────────────
df.to_csv('../data/cleaned/order_reviews_cleaned.csv', index=False)

print(f"Exported: {len(df)} rows")
print("Saved to: data/cleaned/order_reviews_cleaned.csv")

Exported: 98410 rows
Saved to: data/cleaned/order_reviews_cleaned.csv
